In [ ]:
from __future__ import annotations

from datasets import DatasetDict, load_dataset

In [43]:
# "ru"
# ar,
# de,
# zh
lang = "zh"

In [47]:
# ds = load_dataset("DeepPavlov/Multi2WOZ", lang)
ds = load_dataset("DeepPavlov/MultiWOZ-2.1")

In [50]:
def split_dataset_by_domains(ds):
    """Split dataset by domains and filter columns to match each domain.

    Args:
        ds: HuggingFace dataset

    Returns:
        dict: Dictionary with domain names as keys and filtered datasets as values
    """

    def get_domain_from_column(column_name):
        """Extract domain from column name like 'hotel-stars'"""
        if "-" not in column_name:
            return None
        return column_name.split("-")[0]

    def filter_rows_and_columns_for_domain(examples, target_domain):
        """Filter rows that contain the target domain and keep only relevant columns"""
        # Get all column names for this domain
        relevant_columns = ["dialogue_id", "text", "history"]  # Always keep these
        domain_columns = []

        for col in examples.keys():
            col_domain = get_domain_from_column(col)
            if col_domain == target_domain:
                domain_columns.append(col)
                relevant_columns.append(col)

        # Filter rows that have at least one non-"none" value in domain columns
        filtered_examples = {col: [] for col in relevant_columns}

        num_rows = len(examples["dialogue_id"])
        for i in range(num_rows):
            # Check if this row has any non-"none" values for the target domain
            has_domain_data = False
            for domain_col in domain_columns:
                if examples[domain_col][i] != "none":
                    has_domain_data = True
                    break

            # If row has domain data, add it to filtered examples
            if has_domain_data:
                for col in relevant_columns:
                    filtered_examples[col].append(examples[col][i])

        return filtered_examples

    # Split dataset for each domain
    domain_datasets = {}

    # Get all unique domains from column names
    all_domains = set()
    first_ds = ds["train"] if "train" in ds else next(iter(ds.values()))
    for col in first_ds.features.keys():
        domain = get_domain_from_column(col)
        if domain:
            all_domains.add(domain)

    print(f"Found domains: {sorted(all_domains)}")

    for domain_name in sorted(all_domains):
        print(f"Processing domain: {domain_name}")

        # Filter dataset for this domain
        filtered_ds = {}
        for split_name in ds.keys():
            filtered_split = ds[split_name].map(
                lambda examples: filter_rows_and_columns_for_domain(
                    examples, domain_name
                ),
                batched=True,
                remove_columns=list(ds[split_name].features.keys()),
            )

            # Remove empty datasets
            if len(filtered_split) > 0:
                filtered_ds[split_name] = filtered_split
                print(f"  - {split_name}: {len(filtered_split)} rows")

        if filtered_ds:  # Only add if we have data
            domain_datasets[domain_name] = filtered_ds
        else:
            print(f"  - No data found for domain {domain_name}")

    return domain_datasets


# Run the function
domain_datasets = split_dataset_by_domains(ds)

# Print summary
for domain_name, domain_ds in domain_datasets.items():
    print(f"\n{domain_name} domain:")
    for split_name, split_ds in domain_ds.items():
        print(f"  {split_name}: {len(split_ds)} rows, {len(split_ds.features)} columns")
        print(f"    Columns: {list(split_ds.features.keys())}")

Found domains: ['attraction', 'hospital', 'hotel', 'restaurant', 'taxi', 'train']
Processing domain: attraction


Map:   0%|          | 0/56668 [00:00<?, ? examples/s]

  - train: 15690 rows


Map:   0%|          | 0/7374 [00:00<?, ? examples/s]

  - dev: 2396 rows


...

  - train: 17904 rows


Map:   0%|          | 0/7374 [00:00<?, ? examples/s]

  - dev: 2863 rows


Map:   0%|          | 0/7368 [00:00<?, ? examples/s]

  - test: 2949 rows

attraction domain:
  train: 15690 rows, 6 columns
    Columns: ['attraction-area', 'attraction-name', 'attraction-type', 'dialogue_id', 'history', 'text']
  dev: 2396 rows, 6 columns
    Columns: ['attraction-area', 'attraction-name', 'attraction-type', 'dialogue_id', 'history', 'text']
  test: 2433 rows, 6 columns
    Columns: ['attraction-area', 'attraction-name', 'attraction-type', 'dialogue_id', 'history', 'text']

hospital domain:
  train: 384 rows, 4 columns
    Columns: ['hospital-department', 'dialogue_id', 'history', 'text']
  dev: 7 rows, 4 columns
    Columns: ['hospital-department', 'dialogue_id', 'history', 'text']

hotel domain:
  train: 21844 rows, 13 columns
    Columns: ['hotel-area', 'hotel-book day', 'hotel-book people', 'hotel-book stay', 'hotel-internet', 'hotel-name', 'hotel-parking', 'hotel-pricerange', 'hotel-stars', 'hotel-type', 'dialogue_id', 'history', 'text']
  dev: 2769 rows, 13 columns
    Columns: ['hotel-area', 'hotel-book day', 'ho

In [52]:
for domain_name, domain_ds in domain_datasets.items():
    ds_dict = DatasetDict(domain_ds)
    # ds_dict.push_to_hub("DeepPavlov/Multi2WOZ", f"{lang}_{domain_name}")
    ds_dict.push_to_hub("DeepPavlov/MultiWOZ-2.1", domain_name)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/16 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/6.08k [00:00<?, ?B/s]